### learn more about pymupdf ###

In [2]:
import pymupdf
import spacy
import pprint
import pandas as pd
import spacy

In [ ]:
# TOC info

# X'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'

# Z'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_social_bond_framework_series_ad_ae_2019.pdf'
# V'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/ISA_CTM_FRAMEWORK.pdf'
# U'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/SONDA_S_A__GREEN_BOND_FRAMEWORK.pdf'

# Portuguese
# Y'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_Bonds_Framework_Athon_2020.pdf'


In [7]:
# X'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'

pdfX = pymupdf.open('/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf')
# pdfX.get_toc()

# has no TOC
# search of UOP type words
# Let's find out how many have the UoP section heading?
# iterate over pages looking for UOP and return page number
# doesn't seem to be a way of knowing the section heading font
# instead, in how many cases is the next section 'selection and eval of projects'
# how can I isolate just that area of the document - ie that starts at UOP and ends just before S&E of projects


areaUOP = None
areaSEEGP = None
keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Evaluation and Assessment Process']

for page_idx in range(len(pdfX)):
    page = pdfX[page_idx]

    if areaUOP is None:
        for keyword in keywordsUOP:
            start = page.search_for(keyword)
            if start:
                areaUOP = (page_idx, start[0])

    if areaSEEGP is None:
        for keyword in keywordsSEEGP:
            end = page.search_for(keyword)
            if end:
                areaSEEGP = (page_idx, end[0])

print(areaUOP)
print(areaSEEGP)


(1, Rect(85.10399627685547, 407.7648620605469, 171.381591796875, 420.328369140625))
(2, Rect(85.10399627685547, 354.48486328125, 215.33181762695312, 367.0483703613281))


In [364]:
# convert pages to blocks/words

start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0
extractUOPwords = []



for page_idx in range(len(pdfX)):
    page = pdfX[page_idx]
    if page_idx == start_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point:
                extractUOPwords.append(text)
    elif page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y1 < end_point:
                extractUOPwords.append(text)
    elif page_idx > start_page_idx and page_idx < end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            extractUOPwords.append(text)


pdfXwords = extractUOPwords
# need to assess the outcome of the UOP keyword search for the number of consecutive pages, the start point page and the end point page
# for start point page it's start point, for end point page it's end point, for all other pages you don't need the points




In [365]:

# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances']

In [366]:
nlp = spacy.load('en_core_web_lg')

simsre = {}
simsee = {}
simsswwm = {}

re = renewable_energy
ee = energy_efficiency
swwm = sustainable_water_and_wastewater_management
euw = pdfXwords

similarity_threshold = 0.72

for worda in re:
    doca = nlp(worda)
    for wordb in euw:
        docb = nlp(wordb)
        if doca.similarity(docb) >= similarity_threshold:
            sim = doca.similarity(docb)
            simsre[doca[0].text + ' ' + docb[0].text] = sim

for wordc in ee:
    docc = nlp(wordc)
    for wordb in euw:
        docb = nlp(wordb)
        if docc.similarity(docb) >= similarity_threshold:
            sim = docc.similarity(docb)
            simsee[docc[0].text + ' ' + docb[0].text] = sim

for wordd in swwm:
    docd = nlp(wordd)
    for wordb in euw:
        docb = nlp(wordb)
        if docd.similarity(docb) >= similarity_threshold:
            sim = docd.similarity(docb)
            simsswwm[docd[0].text + ' ' + docb[0].text] = sim

dfre = pd.DataFrame.from_dict(simsre, 'index')
dfee = pd.DataFrame.from_dict(simsee, 'index')
dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')




/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/3135857056.py:18: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/3135857056.py:26: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/3135857056.py:34: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:


In [367]:

print('Classification for pdfX')

if dfre.empty:
    print('is not re')
else:
    print('re word pairs')
    print(dfre)
    print('')

if dfee.empty:
    print('is not ee')
else:
    print('ee word pairs')
    print(dfee)
    print('')

if dfswwm.empty:
    print('is not swwm')
else:
    print('swwm word pairs')
    print(dfswwm)
    print('')

Classification for pdfX
re word pairs
                                  0
renewable Renewable        1.000000
renewable renewable        1.000000
solar solar                1.000000
solar photovoltaic         0.790688
power Power                1.000000
transmission Transmission  1.000000
generation generation      0.822127

is not ee
is not swwm


In [368]:
# T:'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_Athon_in_2021.pdf'

pdfT = pymupdf.open('/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_Athon_in_2021.pdf')

areaUOP = None
areaSEEGP = None
keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Evaluation and Assessment Process']


for page_idx in range(len(pdfT)):
    page = pdfT[page_idx]

    if areaUOP is None:
        for keyword in keywordsUOP:
            start = page.search_for(keyword)
            if start:
                areaUOP = (page_idx, start[0])

    if areaSEEGP is None:
        for keyword in keywordsSEEGP:
            end = page.search_for(keyword)
            if end:
                areaSEEGP = (page_idx, end[0])

print(areaUOP)
print(areaSEEGP)

(1, Rect(85.10399627685547, 85.42878723144531, 171.39920043945312, 97.7383804321289))
(1, Rect(85.10399627685547, 432.73876953125, 215.33181762695312, 445.0483703613281))


In [369]:
# convert pages to blocks/words

start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0
extractUOPwords = []



for page_idx in range(len(pdfT)):
    page = pdfT[page_idx]
    # UOP all on a single page
    if start_page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point and y1 < end_point:
                extractUOPwords.append(text)
    # UOP across >1 page
    elif page_idx == start_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point:
                extractUOPwords.append(text)
    elif page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y1 < end_point:
                extractUOPwords.append(text)
    elif page_idx > start_page_idx and page_idx < end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            extractUOPwords.append(text)


pdfTwords = extractUOPwords

In [370]:
simsre = {}
simsee = {}
simsswwm = {}

re = renewable_energy
ee = energy_efficiency
swwm = sustainable_water_and_wastewater_management
euw = pdfTwords

similarity_threshold = 0.72

for worda in re:
    doca = nlp(worda)
    for wordb in euw:
        docb = nlp(wordb)
        if doca.similarity(docb) >= similarity_threshold:
            sim = doca.similarity(docb)
            simsre[doca[0].text + ' ' + docb[0].text] = sim

for wordc in ee:
    docc = nlp(wordc)
    for wordb in euw:
        docb = nlp(wordb)
        if docc.similarity(docb) >= similarity_threshold:
            sim = docc.similarity(docb)
            simsee[docc[0].text + ' ' + docb[0].text] = sim

for wordd in swwm:
    docd = nlp(wordd)
    for wordb in euw:
        docb = nlp(wordb)
        if docd.similarity(docb) >= similarity_threshold:
            sim = docd.similarity(docb)
            simsswwm[docd[0].text + ' ' + docb[0].text] = sim

dfre = pd.DataFrame.from_dict(simsre, 'index')
dfee = pd.DataFrame.from_dict(simsee, 'index')
dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')

/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/252164524.py:16: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/252164524.py:24: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/252164524.py:32: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:


In [371]:
print('Classification for pdfT')

if dfre.empty:
    print('is not re')
else:
    print('re word pairs')
    print(dfre)
    print('')

if dfee.empty:
    print('is not ee')
else:
    print('ee word pairs')
    print(dfee)
    print('')

if dfswwm.empty:
    print('is not swwm')
else:
    print('swwm word pairs')
    print(dfswwm)
    print('')

Classification for pdfT
re word pairs
                              0
renewable renewable    1.000000
solar solar            1.000000
solar photovoltaic     0.790688
solar Solar            1.000000
hydropower hydropower  1.000000
power power            0.849600
generation generation  1.000000
generation Generation  1.000000

is not ee
is not swwm


In [3]:
# Z'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_social_bond_framework_series_ad_ae_2019.pdf'

pdfZ = pymupdf.open('/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Green_social_bond_framework_series_ad_ae_2019.pdf')

areaUOP = None
areaSEEGP = None
keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Evaluation and Assessment Process']


for page_idx in range(len(pdfZ)):
    page = pdfZ[page_idx]

    if areaUOP is None:
        for keyword in keywordsUOP:
            start = page.search_for(keyword)
            if start:
                areaUOP = (page_idx, start[0])

    if areaSEEGP is None:
        for keyword in keywordsSEEGP:
            end = page.search_for(keyword)
            if end:
                areaSEEGP = (page_idx, end[0])

print(areaUOP)
print(areaSEEGP)

(1, Rect(85.03153228759766, 87.50003051757812, 143.58770751953125, 98.54003143310547))
(4, Rect(68.66400146484375, 74.06002807617188, 178.82113647460938, 85.10002899169922))


In [4]:
start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0

noTableMsg = []
hasTableMsg = []
hasDFMsg = []

for page_idx in range(len(pdfZ)):
    page = pdfZ[page_idx]
    # A: UOP all on a single page
    if start_page_idx == end_page_idx:
        tables = page.find_tables(strategy = 'lines_strict')
        if tables.tables:
            bbox = tables[0].bbox
            if bbox[1] > start_point and bbox[3] < end_point:
                tableAheader = tables[0].header.names
                tableAdf = tables[0].to_pandas()
            hasTableMsg.append(tableAheader)
            hasDFMsg.append(tableAdf)
        else:
            noTableMsg.append('No tableA')
            
    # UOP across >1 page
    # B: current page is start page
    elif page_idx == start_page_idx:
        tables = page.find_tables(strategy = 'lines_strict')
        if tables.tables:
            bbox = tables[0].bbox
            if bbox[1] > start_point:
                tableBheader = tables[0].header.names
                tableBdf = tables[0].to_pandas()
            hasTableMsg.append(tableBheader)
            hasDFMsg.append(tableBdf)
        else:
            noTableMsg.append('No tableB')

    # D: current page is neither start nor end page but is in the UOP area
    elif page_idx > start_page_idx and page_idx < end_page_idx:
        tables = page.find_tables(strategy = 'lines_strict')
        if tables.tables:
            tableDheader = tables[0].header.names
            tableDdf = tables[0].to_pandas()
            hasTableMsg.append(tableDheader)
            hasDFMsg.append(tableDdf)
        else:
            noTableMsg.append(page_idx)
            noTableMsg.append('No tableD')

    # C: current page is end page
    elif page_idx == end_page_idx:
        tables = page.find_tables()
        if tables.tables:
            bbox = tables[0].bbox
            if bbox[3] < end_point:
                tableCheader = tables[0].header.names
                tableCdf = tables[0].to_pandas()
            hasTableMsg.append(tableCheader)
            hasDFMsg.append(tableCdf)
        else:
            noTableMsg.append('No tableC')


print(noTableMsg)
print(hasTableMsg)


# this code works only if there is just one table
# check for the 1 table only condition
if len(hasDFMsg) != 1:
    print('More than one table found')

# this over simplifies by assuming the category is always in the first column
if 'Category' in hasTableMsg[0]:
    x = 'singular'
elif 'Categories' in hasTableMsg[0]:
    x = 'plural'
else:
    x = 'not a UOP table or category not in 1st columns'

DFname = hasDFMsg[0]
if x == 'singular':
    uniqueCats = DFname['Category'].unique()
    print(uniqueCats)
elif x == 'plural':
    uniqueCats = DFname['Categories'].unique()
    print(uniqueCats)
else:
    print(x)


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
['No tableB', 2, 'No tableD', 'No tableC']
[['Category', 'Project name', 'Goal']]
<ArrowStringArray>
['Potable Water Supply', 'Resilient infrastructure', 'Sanitation', nan]
Length: 4, dtype: str


In [374]:
# convert pages to blocks/words

start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0
extractUOPwords = []



for page_idx in range(len(pdfZ)):
    page = pdfZ[page_idx]
    # UOP all on a single page
    if start_page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point and y1 < end_point:
                extractUOPwords.append(text)
    # UOP across >1 page
    elif page_idx == start_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point:
                extractUOPwords.append(text)
    elif page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y1 < end_point:
                extractUOPwords.append(text)
    elif page_idx > start_page_idx and page_idx < end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            extractUOPwords.append(text)


pdfZwords = extractUOPwords

In [375]:
simsre = {}
simsee = {}
simsswwm = {}

re = renewable_energy
ee = energy_efficiency
swwm = sustainable_water_and_wastewater_management
euw = pdfZwords

similarity_threshold = 0.72

for worda in re:
    doca = nlp(worda)
    for wordb in euw:
        docb = nlp(wordb)
        if doca.similarity(docb) >= similarity_threshold:
            sim = doca.similarity(docb)
            simsre[doca[0].text + ' ' + docb[0].text] = sim

for wordc in ee:
    docc = nlp(wordc)
    for wordb in euw:
        docb = nlp(wordb)
        if docc.similarity(docb) >= similarity_threshold:
            sim = docc.similarity(docb)
            simsee[docc[0].text + ' ' + docb[0].text] = sim

for wordd in swwm:
    docd = nlp(wordd)
    for wordb in euw:
        docb = nlp(wordb)
        if docd.similarity(docb) >= similarity_threshold:
            sim = docd.similarity(docb)
            simsswwm[docd[0].text + ' ' + docb[0].text] = sim

dfre = pd.DataFrame.from_dict(simsre, 'index')
dfee = pd.DataFrame.from_dict(simsee, 'index')
dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')

/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/4235636271.py:16: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/4235636271.py:24: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/4235636271.py:32: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:


In [376]:
print('Classification for pdfZ')

if dfre.empty:
    print('is not re')
else:
    print('re word pairs')
    print(dfre)
    print('')

if dfee.empty:
    print('is not ee')
else:
    print('ee word pairs')
    print(dfee)
    print('')

if dfswwm.empty:
    print('is not swwm')
else:
    print('swwm word pairs')
    print(dfswwm)
    print('')

Classification for pdfZ
re word pairs
               0
power power  1.0

is not ee
swwm word pairs
                              0
water water            1.000000
water Water            1.000000
potable potable        1.000000
potable Potable        1.000000
wastewater wastewater  1.000000
wastewater Wastewater  1.000000
wastewater effluent    0.818452
sanitation sanitation  1.000000
sanitation Sanitation  1.000000
treatment treatment    1.000000
treatment Treatment    1.000000



In [8]:
# V'/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/ISA_CTM_FRAMEWORK.pdf'
pdfV = pymupdf.open('/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/ISA_CTM_FRAMEWORK.pdf')

areaUOP = None
areaSEEGP = None
keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Process for Project Evaluation and Selection', 'Project Evaluation and Assessment Process']

for page_idx in range(len(pdfV)):
    page = pdfV[page_idx]

    if areaUOP is None:
        for keyword in keywordsUOP:
            start = page.search_for(keyword)
            if start:
                areaUOP = (page_idx, start[0])

    if areaSEEGP is None:
        for keyword in keywordsSEEGP:
            end = page.search_for(keyword)
            if end:
                areaSEEGP = (page_idx, end[0])
            # else:
                # last_page_words = pdfV[-1].get_text("words")
                # areaSEEGP = (pdfV[-1].number, last_page_words[-20])

print(areaUOP)
print(areaSEEGP)
print(type(areaUOP))
print(type(areaSEEGP))


(3, Rect(120.9800033569336, 186.52999877929688, 210.60275268554688, 197.5699920654297))
(5, Rect(141.6199951171875, 232.73001098632812, 359.7041320800781, 243.77000427246094))
<class 'tuple'>
<class 'tuple'>


In [9]:
start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0

noTableMsg = []
hasTableMsg = []
hasDFMsg = []

for page_idx in range(len(pdfV)):
    page = pdfV[page_idx]
    # A: UOP all on a single page
    if start_page_idx == end_page_idx:
        tables = page.find_tables(strategy = 'lines_strict')
        if tables.tables:
            bbox = tables[0].bbox
            if bbox[1] > start_point and bbox[3] < end_point:
                tableAheader = tables[0].header.names
                tableAdf = tables[0].to_pandas()
                hasTableMsg.append(tableAheader)
                hasDFMsg.append(tableAdf)
        else:
            noTableMsg.append('No tableA')
            
    # UOP across >1 page
    # B: current page is start page
    elif page_idx == start_page_idx:
        tables = page.find_tables(strategy = 'lines_strict')
        if tables.tables:
            bbox = tables[0].bbox
            if bbox[1] > start_point:
                tableBheader = tables[0].header.names
                tableBdf = tables[0].to_pandas()
                hasTableMsg.append(tableBheader)
                hasDFMsg.append(tableBdf)
        else:
            noTableMsg.append('No tableB')

    # D: current page is neither start nor end page but is in the UOP area
    elif page_idx > start_page_idx and page_idx < end_page_idx:
        tables = page.find_tables(strategy = 'lines_strict')
        if tables.tables:
            tableDheader = tables[0].header.names
            if 'Category' in tableDheader or 'Categories' in tableDheader or 'Eligible Project Category' in tableDheader:
                tableDdf = tables[0].to_pandas()
                hasTableMsg.append(tableDheader)
                hasDFMsg.append(tableDdf)
        else:
            noTableMsg.append(page_idx)
            noTableMsg.append('No tableD')

    # C: current page is end page
    elif page_idx == end_page_idx:
        tables = page.find_tables()
        if tables.tables:
            bbox = tables[0].bbox
            if bbox[3] < end_point:
                tableCheader = tables[0].header.names
                if 'Category' in tableCheader or 'Categories' in tableCheader or 'Eligible Project Category' in tableCheader:
                    tableCdf = tables[0].to_pandas()
                    hasTableMsg.append(tableCheader)
                    hasDFMsg.append(tableCdf)
        else:
            noTableMsg.append('No tableC')


print(noTableMsg)
print(hasTableMsg)


# this code works only if there is just one table
# check for the 1 table only condition
if len(hasDFMsg) != 1:
    print('More than one table found')

# this over simplifies by assuming the category is always in the first column

''' the header condition code oversimplifies by assuming table fragments 
split across pages without a header row contain no new category labels'''

if 'Category' in hasTableMsg[0]:
    x = 'singular'
elif 'Categories' in hasTableMsg[0]:
    x = 'plural'
elif 'Eligible Project Category':
    x = 'phrase'
else:
    x = 'not a UOP table or category not in 1st columns'

DFname = hasDFMsg[0]
if x == 'singular':
    uniqueCats = DFname['Category'].unique()
    print(uniqueCats)
elif x == 'plural':
    uniqueCats = DFname['Categories'].unique()
    print(uniqueCats)
elif x == 'phrase':
    uniqueCats = DFname['Eligible Project Category'].unique()
    print(uniqueCats)
else:
    print(x)


    

['No tableB']
[['Eligible Project Category', 'Questions', 'Alignment to the UN SDGs']]
<ArrowStringArray>
['Renewable Energy', 'Energy Efficiency']
Length: 2, dtype: str


In [379]:
# convert pages to blocks/words

start_page_idx = areaUOP[0]
end_page_idx = areaSEEGP[0]
start_point = areaUOP[1].y1
end_point = areaSEEGP[1].y0


extractUOPwords = []


for page_idx in range(len(pdfV)):
    page = pdfV[page_idx]
    # UOP all on a single page
    if start_page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point and y1 < end_point:
                extractUOPwords.append(text)
    # UOP across >1 page
    elif page_idx == start_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y0 > start_point:
                extractUOPwords.append(text)
    elif page_idx == end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            if y1 < end_point:
                extractUOPwords.append(text)
    elif page_idx > start_page_idx and page_idx < end_page_idx:
        for word in page.get_text('words'):
            x0, y0, x1, y1, text, *_ = word
            extractUOPwords.append(text)


pdfVwords = extractUOPwords

In [380]:
simsre = {}
simsee = {}
simsswwm = {}

re = renewable_energy
ee = energy_efficiency
swwm = sustainable_water_and_wastewater_management
euw = pdfVwords

similarity_threshold = 0.72

for worda in re:
    doca = nlp(worda)
    for wordb in euw:
        docb = nlp(wordb)
        if doca.similarity(docb) >= similarity_threshold:
            sim = doca.similarity(docb)
            simsre[doca[0].text + ' ' + docb[0].text] = sim

for wordc in ee:
    docc = nlp(wordc)
    for wordb in euw:
        docb = nlp(wordb)
        if docc.similarity(docb) >= similarity_threshold:
            sim = docc.similarity(docb)
            simsee[docc[0].text + ' ' + docb[0].text] = sim

for wordd in swwm:
    docd = nlp(wordd)
    for wordb in euw:
        docb = nlp(wordb)
        if docd.similarity(docb) >= similarity_threshold:
            sim = docd.similarity(docb)
            simsswwm[docd[0].text + ' ' + docb[0].text] = sim

dfre = pd.DataFrame.from_dict(simsre, 'index')
dfee = pd.DataFrame.from_dict(simsee, 'index')
dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')

/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/3266279397.py:16: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if doca.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/3266279397.py:24: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docc.similarity(docb) >= similarity_threshold:
/var/folders/pf/t466z2553r9f82s2v7n0pgfr0000gn/T/ipykernel_3369/3266279397.py:32: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  if docd.similarity(docb) >= similarity_threshold:


In [381]:
print('Classification for pdfV')

if dfre.empty:
    print('is not re')
else:
    print('re word pairs')
    print(dfre)
    print('')

if dfee.empty:
    print('is not ee')
else:
    print('ee word pairs')
    print(dfee)
    print('')

if dfswwm.empty:
    print('is not swwm')
else:
    print('swwm word pairs')
    print(dfswwm)
    print('')

Classification for pdfV
re word pairs
                                  0
renewable Renewable        1.000000
renewable renewable        1.000000
solar solar                0.856726
wind wind                  0.844071
grid grid                  1.000000
transmission transmission  1.000000
generation generation      1.000000

ee word pairs
                         0
efficiency Efficiency  1.0
efficiency efficiency  1.0

is not swwm


In [124]:
!python3 -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.7 MB/s  0:02:210:00:0100:04
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [134]:

nlpA = spacy.load('en_core_web_lg')
texta = 'we are financing a solar power plant'
textb = 'renewable energy projects qualify the bond as green'
doca = nlpA(texta)
docb = nlpA(textb)
tokena = doca[4]
tokenb = docb[0]
print(tokena.similarity(tokenb))
print(doca.similarity(docb))

0.6459308862686157
0.8404058814048767


In [135]:
texta = 'solar'
textb = 'renewable'
doca = nlpA(texta)
docb = nlpA(textb)
print(tokena.similarity(tokenb))

0.6459308862686157
